# 13.4 Where the Memory Actually Goes

**Prerequisites:** 13.1 Objects, Names and the Heap, 13.3 Reference Counting, 5.1 Python OOPs, 5.3 Dataclasses  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 **`sys.getsizeof` is shallow** — what it measures, and what it silently omits
- The per-object header every Python object pays
- Container overhead, and why a list over-allocates as it grows
- 🔴 **The same record stored seven ways**, measured at 100,000 rows
- `__slots__` measured rather than described — and exactly what it costs you
- When any of this is worth acting on, and when it is a distraction

---

## The question this answers

**13.1** established that everything is a heap object. The practical follow-up is: *how much
does that cost?* A service holding a hundred thousand task records in memory has a very
different footprint depending on how those records are represented — and the difference is
large enough to decide whether the process fits in its container.

This notebook is about **bytes**. It is deliberately not about speed: `timeit`, `cProfile` and
`tracemalloc` belong to **17.5**, and operation *complexity* belongs to **14.1** and **14.2**.

> The measurements below are from **CPython on a 64-bit build**. Every number changes on a
> 32-bit build, on PyPy, or between CPython releases. The *ratios* are what survive.

## 🔴 `getsizeof` does not measure what you think

`sys.getsizeof(obj)` returns the size of **that object alone**. For any container, that means
the size of the container's own machinery plus a *pointer* per element — not the elements.

In [ ]:
import sys

task_ids = [f"task-{n:05d}" for n in range(1000)]

shallow = sys.getsizeof(task_ids)
contents = sum(sys.getsizeof(item) for item in task_ids)

print(f"  sys.getsizeof(list of 1,000 ids) : {shallow:>9,} bytes")
print(f"  the 1,000 id strings themselves  : {contents:>9,} bytes")
print(f"  actual total                     : {shallow + contents:>9,} bytes")
print(f"\n  getsizeof reported {(shallow + contents) / shallow:.1f}x less than the real cost.")

print("\n  The list stores 1,000 pointers, 8 bytes each:")
print(f"    1,000 x 8            = {1000 * 8:>7,} bytes of pointers")
print(f"    + list object header = {shallow - 8000:>7,} bytes")
print(f"    = getsizeof          = {shallow:>7,} bytes")

The list itself accounts for a small fraction of the memory actually
consumed. It holds a thousand **pointers**; the strings they point at are separate heap objects
that `getsizeof` never visits.

🔴 **There is no `deep_getsizeof` in the standard library**, and writing a correct one is
harder than it looks — you must track already-seen objects by `id()` or a shared string gets
counted many times, and cycles (**13.3**) make a naive version recurse forever.

For real measurement use **`tracemalloc`** (**17.5**), which records allocations rather than
trying to walk the object graph.

## Every object pays a header

An object is not just its data. It carries a reference count and a type pointer at minimum,
which is why even an empty container is not free.

In [ ]:
print("empty containers are not empty:")
for label, empty in [("tuple", ()), ("list", []), ("dict", {}), ("set", set()),
                     ("str", ""), ("int", 0), ("object()", object())]:
    print(f"  {label:10} {sys.getsizeof(empty):>4,} bytes")

print("\nand a list over-allocates so that append() stays amortised O(1):")
growing, previous, jumps = [], sys.getsizeof([]), []
for n in range(70):
    growing.append(n)
    size = sys.getsizeof(growing)
    if size != previous:
        jumps.append((len(growing), size))
        previous = size

print("  length at which the list resized, and its new size:")
for length, size in jumps[:8]:
    print(f"    {length:>3} items -> {size:>5,} bytes")
print("\n  It grows in geometric steps, not one slot at a time - which is exactly")
print("  where 14.1's amortised O(1) append comes from. (14.6 does the same")
print("  for dict and set, from the hashing side.)")

A bare `object()` still costs bytes, and an empty `set` costs several
times an empty `dict`. The list growth pattern is the amortisation argument from **14.1** made
visible: capacity jumps in widening steps so that most `append` calls are free, and the
occasional one pays for a reallocation.

**14.6** shows the equivalent for `dict` and `set` from the hashing angle. The point here is
narrower: **capacity is not length**, so a list that once held a million items still owns a
large buffer after you remove them.

## 🔴 The measurement that matters: one record, seven ways

This is the decision that shows up in real services. A task record has three fields. You can
hold it as a dict, a tuple, a namedtuple, a plain class, a dataclass, or either of the last two
with `__slots__`.

`__slots__` was introduced in **5.1** and `dataclass(slots=True)` in **5.3**. Here they get
measured.

In [ ]:
from collections import namedtuple
from dataclasses import dataclass

RECORDS = 100_000

TaskRow = namedtuple("TaskRow", "id state attempts")


class TaskPlain:
    def __init__(self, id, state, attempts):
        self.id, self.state, self.attempts = id, state, attempts


class TaskSlotted:
    __slots__ = ("id", "state", "attempts")

    def __init__(self, id, state, attempts):
        self.id, self.state, self.attempts = id, state, attempts


@dataclass
class TaskDataclass:
    id: str
    state: str
    attempts: int


@dataclass(slots=True)
class TaskDataclassSlots:
    id: str
    state: str
    attempts: int


def record_size(build):
    """Object itself, plus its instance __dict__ when it has one."""
    obj = build("task-00001", "queued", 0)
    size = sys.getsizeof(obj)
    instance_dict = getattr(obj, "__dict__", None)
    if instance_dict is not None:
        size += sys.getsizeof(instance_dict)
    return size


candidates = [
    ("dict",                  lambda i, s, a: {"id": i, "state": s, "attempts": a}),
    ("tuple",                 lambda i, s, a: (i, s, a)),
    ("namedtuple",            TaskRow),
    ("class with __dict__",   TaskPlain),
    ("dataclass",             TaskDataclass),
    ("class with __slots__",  TaskSlotted),
    ("dataclass(slots=True)", TaskDataclassSlots),
]

measured = [(label, record_size(build)) for label, build in candidates]
worst = max(size for _, size in measured)
best = min(size for _, size in measured)

print(f"one task record, and what {RECORDS:,} of them cost:\n")
print(f"  {'representation':24}{'bytes':>7}{'at 100k':>11}")
print("  " + "-" * 56)
for label, size in measured:
    mib = size * RECORDS / 1_048_576
    print(f"  {label:24}{size:>7,}{mib:>9.1f} MiB  {'#' * int(size / worst * 20)}")

print(f"\n  worst / best = {worst / best:.1f}x")
print(f"  difference at {RECORDS:,} records = {(worst - best) * RECORDS / 1_048_576:.0f} MiB")

The worst/best ratio is printed above, and at a hundred thousand
records the gap runs to tens of megabytes — the difference between comfortable and OOM-killed
in a small container.

Read the table in three groups:

- **`class with __dict__` and plain `dataclass` are the most expensive.** Each instance carries
  its own dictionary, and that dictionary dominates the cost.
- **`tuple` and `namedtuple` are cheap** because there is no per-instance dict and no slack.
  `namedtuple` gives you field names for free at exactly the same size.
- 🔴 **`__slots__` wins outright**, and `dataclass(slots=True)` is identical to a hand-written
  slotted class — you give up nothing to get the generated `__init__`, `__repr__` and `__eq__`.

**The practical rule:** if you will hold more than a few thousand instances of a class,
`@dataclass(slots=True)` is close to a free win. Below that, it is noise and readability should
decide.

## What `__slots__` actually costs

It is not free. Declaring slots removes the instance dictionary, and three things go with it.

In [ ]:
import weakref

task = TaskSlotted("task-00001", "queued", 0)

print("1. no instance __dict__:")
print("     hasattr(task, '__dict__') :", hasattr(task, "__dict__"))

print("\n2. you cannot add attributes that were not declared:")
try:
    task.assigned_to = "team-a"
except AttributeError as exc:
    print("     AttributeError:", exc)

print("\n3. it is not weak-referenceable (13.3 needs this!):")
try:
    weakref.ref(task)
except TypeError as exc:
    print("     TypeError:", exc)


class TaskWeakable:
    """Add '__weakref__' to the slots to get weak references back."""
    __slots__ = ("id", "state", "attempts", "__weakref__")

    def __init__(self, id, state, attempts):
        self.id, self.state, self.attempts = id, state, attempts


fixed = TaskWeakable("task-00002", "queued", 0)
print("\n     with '__weakref__' in __slots__:", weakref.ref(fixed) is not None)
print(f"     and it still costs only {sys.getsizeof(fixed):,} bytes")

All three limitations are the same limitation: **there is no instance
dictionary**, so there is nowhere to put an undeclared attribute and nowhere to store a weak
reference.

The third one is the trap, and it connects directly to **13.3** — a `WeakValueDictionary`
cache silently becomes unusable the moment you add `__slots__` to the cached class. The fix is
one entry: `"__weakref__"`.

⚠️ **Two more things to know.** Slots are not inherited usefully: if any base class lacks
`__slots__`, instances get a `__dict__` anyway and the saving evaporates. And a slotted class
cannot use class-level defaults for slotted names, because the slot descriptor (**13.5**)
occupies that name on the class.

## When to act on any of this

| Situation | Worth measuring? |
|---|---|
| A handful of objects | no — readability wins |
| 🔴 Tens of thousands of records held at once | yes — `slots=True` is close to free |
| A cache or registry that grows | 🔴 the leak matters far more than the size (**13.3**) |
| Large numeric arrays | stop measuring Python objects; use `array`, `numpy` or `memoryview` |
| Millions of repeated strings | `sys.intern` (**13.1**) can be a real win |
| "The process feels heavy" | measure with `tracemalloc` (**17.5**) before changing anything |

🔴 **The largest memory win is almost never the representation.** It is not holding the data at
all — streaming with a generator (**4.3**) instead of building a list, or letting the database
do the aggregation (**10.1**). Choose the representation after you have decided you genuinely
need the objects in memory.

---

## Common Mistakes & Pitfalls

1. 🔴 **Trusting `sys.getsizeof` on a container.** It counts pointers, not the objects they point at.
2. **Writing a recursive deep-size function without tracking `id()`s.** Shared objects are counted repeatedly and cycles never terminate (**13.3**).
3. **Assuming a list releases memory when you remove items.** Capacity is not length.
4. 🔴 **Adding `__slots__` to a class that a `WeakValueDictionary` caches.** It stops being weak-referenceable until you add `"__weakref__"`.
5. **Expecting `__slots__` to help when a base class lacks it.** Instances get a `__dict__` regardless.
6. **Micro-optimising representation for a few hundred objects.** The saving is invisible; the readability cost is not.
7. **Comparing byte counts across machines or versions.** Compare ratios, on one build.
8. **Reaching for `__slots__` before checking whether the data needs to be in memory at all.**

## Best Practices

- Use `@dataclass(slots=True)` by default for record types you will hold in bulk (**5.3**).
- Measure with `tracemalloc` (**17.5**); use `getsizeof` only for single objects you understand.
- Prefer `namedtuple` for small immutable rows — same size as a tuple, with names.
- Add `"__weakref__"` to `__slots__` whenever the object may be cached weakly (**13.3**).
- Stream with generators (**4.3**) instead of materialising lists you only iterate once.
- For homogeneous numbers, leave Python objects behind: `array`, `numpy`, or `memoryview` (**8.5**).
- Bound the collection before optimising the element — an unbounded cache dwarfs any per-object saving.

## Practice Exercises

Try these before moving on.

1. Measure a dict, a namedtuple and a slotted dataclass holding the *same* five fields. Does the ranking from the notebook hold at five fields?
2. 🔴 Write a `deep_size(obj)` that tracks seen `id()`s. Test it on a structure that shares a sublist twice, and on one containing a cycle.
3. Build a list of a million ints, delete every element, and show with `getsizeof` that the list has not shrunk. Then show what does shrink it.
4. Add `__slots__` to a class, then try to cache it in a `WeakValueDictionary`. Fix the error and confirm the size barely changed.
5. Take a real record type from **19.1** or **19.3** and convert it to `slots=True`. Measure the difference at your actual row count.
6. Compare a list of 100,000 identical strings before and after `sys.intern` (**13.1**). Where did the saving come from — the list or the strings?
7. 🔴 Load a CSV (**8.2**) into a list of dicts, then into a list of slotted dataclasses. Measure both with `tracemalloc` (**17.5**), not `getsizeof`. Do the ratios match this notebook?
8. **Interview question:** *“How would you reduce the memory of a service holding a million records?”* Give the answer in the order you would actually try things.

---

## Version notes

| Version | Change |
|---|---|
| **3.3** | Key-sharing instance dictionaries (PEP 412) — the reason a `__dict__` class is not *even* worse than measured here |
| **3.10** | `dataclasses` gained `slots=True`; before that you wrote `__slots__` by hand |
| **3.11** | Objects got a smaller header as part of the "Faster CPython" work; absolute sizes shifted |
| **3.12** | 🔴 Immortal objects (**13.1**) — the refcount field still exists but never changes for them |
| **3.12** | Instance dictionaries were reworked again; measure on the version you deploy |

> 🔴 **These are CPython, 64-bit numbers.** A 32-bit build roughly halves pointer sizes; PyPy
> lays objects out completely differently and `getsizeof` may not even be meaningful there.
> Take the ratios, never the absolute figures.

## 13 How Python Works Under the Hood — the folder

| Notebook | Covers |
|---|---|
| **13.1** | objects on the heap, names, identity, interning, immortality |
| **13.2** | the call stack, frames, recursion limits, tracebacks |
| **13.3** | reference counting, cycles, the collector, `weakref` |
| **13.4** | this notebook — object sizes, container overhead, `__slots__` measured |
| **13.5** | attribute lookup and class creation — descriptors, `__init_subclass__`, metaclasses |

**The one-sentence version:** *`getsizeof` measures one object, every object pays a header, and
for anything you hold in bulk `slots=True` is the cheapest win available.*

## Related

- **5.1 / 5.3** — where `__slots__` and `dataclass(slots=True)` are introduced
- **13.3** — why a leak matters more than a representation
- **14.1 / 14.2** — amortised growth and operation costs
- **14.6** — `dict` and `set` internals from the hashing side
- **17.5** — `tracemalloc`, the tool for measuring memory properly